# Few-Shot Prompting — Llama-3.2-1B-Instruct

**Strategy:** Few-shot — cung cấp 4 ví dụ đã labeled (2 safe + 2 defective)
trước query. Giúp model nhận ra pattern và output format.

**Model:** `unsloth/Llama-3.2-1B-Instruct` (1B params)

**Dataset:** `code_x_glue_cc_defect_detection` test set (2,732 samples)

## 1. Setup

In [ ]:
import os

# --- Colab auto-setup ---
IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
REPO_URL = "https://github.com/MinhTuan2405/IE105.21_SBugDwLLM.git"
BRANCH = "trung"
NOTEBOOK_SUBDIR = "prompt/llama32"

if IN_COLAB:
    REPO_DIR = f"/content/IE105.21_SBugDwLLM"
    if not os.path.exists(REPO_DIR):
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    os.chdir(os.path.join(REPO_DIR, NOTEBOOK_SUBDIR))
    print(f"Working directory: {os.getcwd()}")

!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q scikit-learn matplotlib seaborn huggingface_hub

from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if hf_token:
    login(token=hf_token)
    print("Logged in with HF_TOKEN")
else:
    login()  # Fallback: interactive prompt

In [ ]:
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from utils.data_loader import load_defect_dataset, build_chat_messages, get_few_shot_examples
from utils.evaluation import evaluate_predictions, parse_model_output

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Model

In [ ]:
MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH = 2048  # longer to fit few-shot examples
RESULTS_DIR = "../../docs/llama32_docs"
NUM_EXAMPLES = 4  # 2 safe + 2 defective

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model.eval()
print(f"Model loaded. Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

## 3. Load Dataset & Prepare Few-Shot Examples

In [ ]:
train_data, _, test_data = load_defect_dataset()
print(f"Test samples: {len(test_data)}")

few_shot_examples = get_few_shot_examples(train_data, n=NUM_EXAMPLES, seed=42)
print(f"\nFew-shot examples ({len(few_shot_examples)}):")
for i, (code, label) in enumerate(few_shot_examples):
    label_str = "Defective" if label == 1 else "Safe"
    print(f"  Example {i+1}: {label_str} (label={label}), {len(code)} chars")

## 4. Few-Shot Inference

In [ ]:
# Preview prompt
sample_messages = build_chat_messages(test_data[0]["func"], examples=few_shot_examples)
sample_prompt = tokenizer.apply_chat_template(sample_messages, tokenize=False, add_generation_prompt=True)
print(f"Prompt length: {len(sample_prompt)} chars")
print(sample_prompt[:1000])
print("...")

In [ ]:
device = next(model.parameters()).device
y_true, y_pred, failed_parses = [], [], []
truncated_count = 0

for i, sample in enumerate(tqdm(test_data, desc="Few-shot inference")):
    messages = build_chat_messages(sample["func"], examples=few_shot_examples)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)

    full_len = len(tokenizer.encode(prompt))
    if full_len > MAX_SEQ_LENGTH:
        truncated_count += 1

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False)

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred = parse_model_output(generated)

    if pred == -1:
        failed_parses.append({"index": i, "output": generated, "label": int(sample["target"])})
        pred = 0

    y_true.append(int(sample["target"]))
    y_pred.append(pred)

print(f"\nFailed to parse: {len(failed_parses)} / {len(test_data)}")
print(f"Truncated prompts: {truncated_count} / {len(test_data)}")
if failed_parses[:5]:
    print("Sample failed:")
    for fp in failed_parses[:5]:
        print(f"  idx={fp['index']}, output='{fp['output']}'")

## 5. Evaluation

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

metrics = evaluate_predictions(
    y_true, y_pred,
    model_name="llama32",
    strategy="few_shot",
    save_dir=RESULTS_DIR,
)

## 6. Error Analysis

In [ ]:
errors = [
    {"index": i, "true": yt, "pred": yp, "code": test_data[i]["func"][:200]}
    for i, (yt, yp) in enumerate(zip(y_true, y_pred))
    if yt != yp
]

false_positives = [e for e in errors if e["true"] == 0 and e["pred"] == 1]
false_negatives = [e for e in errors if e["true"] == 1 and e["pred"] == 0]

print(f"Total errors: {len(errors)}")
print(f"False Positives: {len(false_positives)}")
print(f"False Negatives: {len(false_negatives)}")

print("\n--- Sample False Negatives ---")
for e in false_negatives[:3]:
    print(f"  idx={e['index']}: {e['code'][:100]}...")

print("\n--- Sample False Positives ---")
for e in false_positives[:3]:
    print(f"  idx={e['index']}: {e['code'][:100]}...")

## 7. Prompt Template Used

In [ ]:
print("=" * 60)
print("FEW-SHOT PROMPT TEMPLATE")
print("=" * 60)
template = tokenizer.apply_chat_template(
    build_chat_messages("<CODE_HERE>", examples=few_shot_examples),
    tokenize=False, add_generation_prompt=True,
)
print(template)

## 8. Compare All Prompting Strategies

Load tất cả kết quả prompting để so sánh.

In [ ]:
import json
import glob

result_files = glob.glob(os.path.join(RESULTS_DIR, "llama32_*.json"))
if result_files:
    print("\n" + "="*60)
    print("  ALL RESULTS")
    print("="*60)
    for f in sorted(result_files):
        with open(f) as fp:
            r = json.load(fp)
        print(f"\n{r['strategy']}:")
        for k, v in r['metrics'].items():
            print(f"  {k:>12s}: {v:.4f}")